In [0]:
def load_data_to_raw_traffic():
    from pyspark.sql import functions as F
    from pyspark.sql.types import StructField,StructType,IntegerType,StringType,DoubleType
    #from pyspark.sql import current_timeStamp
    source = spark.sql('DESCRIBE External Location trafficprojectexternallocation').select('url').collect()[0][0]
    schema = StructType([
    StructField("Record_ID", IntegerType()),
    StructField("Count_point_id", IntegerType()),
    StructField("Direction_of_travel", StringType()),
    StructField("Year", IntegerType()),
    StructField("Count_date", StringType()),
    StructField("hour", IntegerType()),
    StructField("Region_id", IntegerType()),
    StructField("Region_name", StringType()),
    StructField("Local_authority_name", StringType()),
    StructField("Road_name", StringType()),
    StructField("Road_Category_ID", IntegerType()),
    StructField("Start_junction_road_name", StringType()),
    StructField("End_junction_road_name", StringType()),
    StructField("Latitude", DoubleType()),
    StructField("Longitude", DoubleType()),
    StructField("Link_length_km", DoubleType()),
    StructField("Pedal_cycles", IntegerType()),
    StructField("Two_wheeled_motor_vehicles", IntegerType()),
    StructField("Cars_and_taxis", IntegerType()),
    StructField("Buses_and_coaches", IntegerType()),
    StructField("LGV_Type", IntegerType()),
    StructField("HGV_Type", IntegerType()),
    StructField("EV_Car", IntegerType()),
    StructField("EV_Bike", IntegerType())
    ])

    df = (
        spark.readStream
            .format('CloudFiles')
            .option('CloudFiles.format','CSV')
            .schema(schema)
            .option('header','True')
            .option('CloudFiles.schemaLocation',f'{source}/Checkpoint/Raw_Traffic_Load/Infer_Schema')
            .load(f'{source}/Landing/Raw_Traffics')
            .withColumn('Extract_Time',F.current_timestamp())
    )
    return df


In [0]:
def write_to_raw_traffic(df):
    source = spark.sql('DESCRIBE External Location trafficprojectexternallocation').select('url').collect()[0][0]
    ws = (
        df.writeStream
          .format('delta')
          #.option()
          .option('CheckPointLocation', f'{source}/Checkpoint/Raw_Traffic_Load/Checkpoint')
          .trigger(availableNow = True)
          .toTable('traffic_data.bronze.raw_traffic')
    )

In [0]:
df = load_data_to_raw_traffic()
write_to_raw_traffic(df)

In [0]:
data = spark.sql("select * from traffic_data.bronze.raw_traffic where Extract_Time is NULL ")
display(data)

In [0]:
df = (spark.readStream
        .format("CloudFiles")
        .option("CloudFiles.format","CSV")
        .option('header','True')
        .option('CloudFiles.schemaLocation', f"""{source}/CheckPoint/rawTrafficLoad/schemaInfer""")
        #.trigger(availableNow = True)
        .load(f"""{source}/Landing/raw_traffics""")
)

In [0]:
#display(df, checkpointLocation=f"{source}/CheckPoint/rawTrafficLoad/displayCheckpoint")

In [0]:
df_clean = (
    df.select([
        F.col(c).alias(
            c.strip()
            .lower()
            .replace(" ", "_")
            #.replace("-","_")
        ) for c in df.columns
    ])
)

In [0]:
wd = (df_clean.writeStream
        .format('delta')
        .trigger(availableNow = True)
        .option('mergeSchema', 'True')
        .option('checkPointLocation', f"""{source}/CheckPoint/rawTrafficLoad/CheckPoint""")
        .toTable('traffic_data.bronze.raw_traffic')
)

In [0]:
data = spark.sql("select * from traffic_data.bronze.raw_traffic")
display(data)

In [0]:
df.select("*", *[lit(None).alias(i)] for i in rang(1,101))